# Experiment 11 — Interactive Generative AI Application using Gradio
**Aim:** Develop an interactive application using a pre-trained LLM and expose prompt input, generation controls and safe error handling.


# Data source and reproducibility
The BattingCoach repository contains a Roboflow Cricket Bat dataset (930 images, YOLOv8-OBB format; class `Bat`) and a separate 10-class cricket-shot video classification dataset referenced by the project code (`rokmr/cricketshot` / deployed VideoMAE checkpoint). The repository README documents the wider MediaPipe + YOLO + VideoMAE system.

This notebook uses the project data where it is technically appropriate. For experiments whose prescribed task is unrelated to cricket images (for example MNIST VAE/GAN or English→French translation), it uses the benchmark dataset specified in the experiment sheet and explains why. Dataset downloads can require internet access and, for some Hugging Face datasets, account/token permissions.

**Repository:** https://github.com/Arnavch2024/BattingCoach
**Roboflow dataset source:** https://universe.roboflow.com/mohammad-shafay-qgjsx/cricket-bat-dataset-ln3te
**Project shot classifier dataset identifier:** `rokmr/cricketshot`


### Dataset/model mapping
The UI uses a small pre-trained instruction model and BattingCoach-domain prompts. The repository itself has a Gradio interface for its cricket-coaching backend; this notebook creates a separate educational LLM demo and does not overwrite or modify the project application.

## 1. Install Gradio and Transformers

In [1]:
import importlib.metadata
from importlib.metadata import version

try:
    print('gradio:', version('gradio'))
except importlib.metadata.PackageNotFoundError:
    print('gradio: not installed')
try:
    print('transformers:', version('transformers'))
except importlib.metadata.PackageNotFoundError:
    print('transformers: not installed')
try:
    print('torch:', version('torch'))
except importlib.metadata.PackageNotFoundError:
    print('torch: not installed')
try:
    print('accelerate:', version('accelerate'))
except importlib.metadata.PackageNotFoundError:
    print('accelerate: not installed')


gradio: 6.31.0
transformers: 5.22.1
torch: 2.15.0
accelerate: 1.15.0


## 2. Load the pre-trained model

In [2]:
import torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("Loaded",MODEL_ID)

C:\Users\nikhi\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Loading weights:  12%|█▏        | 35/290 [00:00<00:00, 311.05it/s]


Loading weights:  23%|██▎       | 67/290 [00:00<00:00, 233.53it/s]


Loading weights:  32%|███▏      | 92/290 [00:00<00:00, 232.82it/s]

... (8 outputs left out)


Loading weights:  88%|████████▊ | 256/290 [00:01<00:00, 139.89it/s]


Loading weights:  93%|█████████▎| 271/290 [00:01<00:00, 125.50it/s]


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 167.92it/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct


## 3. Implement prompt validation and generation

In [18]:
def generate_response(prompt, max_new_tokens=180, temperature=0.7):
    if prompt is None or not prompt.strip():
        return "Please enter a prompt."
    if len(prompt) > 4000:
        return "Prompt is too long. Please keep it below 4,000 characters."
    try:
        system = ("You are an educational cricket batting assistant. "
                  "Give general technique information, avoid medical diagnosis, "
                  "and clearly state when information is uncertain.")
        messages=[{"role":"system","content":system},{"role":"user","content":prompt}]
        chat=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        inputs=tokenizer(chat,return_tensors="pt").to(model.device)
        with torch.no_grad():
            out=model.generate(**inputs,max_new_tokens=int(max_new_tokens),
                do_sample=float(temperature)>0,
                temperature=max(float(temperature),0.01),
                top_p=0.9,pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)
    except Exception as exc:
        return f"Generation failed ({type(exc).__name__}). Check model memory and runtime."

## 4. Build the Gradio interface

In [4]:
with gr.Blocks(title="BatCoach GenAI Assistant") as demo:
    gr.Markdown("# BatCoach GenAI Assistant\nEducational demonstration using a pre-trained LLM.")
    prompt=gr.Textbox(label="Ask a cricket batting question", lines=4,
                      placeholder="Explain three checks for a cover drive...")
    with gr.Row():
        max_tokens=gr.Slider(32,256,value=160,step=16,label="Maximum new tokens")
        temperature=gr.Slider(0.0,1.2,value=0.7,step=0.1,label="Temperature")
    submit=gr.Button("Generate answer")
    output=gr.Textbox(label="Generated response",lines=8)
    submit.click(generate_response,inputs=[prompt,max_tokens,temperature],outputs=output)
    prompt.submit(generate_response,inputs=[prompt,max_tokens,temperature],outputs=output)
demo

Gradio Blocks instance: 2 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x000001AC78B79A90>
 |-<gradio.components.slider.Slider object at 0x000001AC78B79D30>
 |-<gradio.components.slider.Slider object at 0x000001AC78BC8050>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x000001AC78BC8190>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x000001AC78B79A90>
 |-<gradio.components.slider.Slider object at 0x000001AC78B79D30>
 |-<gradio.components.slider.Slider object at 0x000001AC78BC8050>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x000001AC78BC8190>

## 5. Launch the application

In [17]:
# In Colab, share=True creates a temporary public link. Do not enter confidential information.
demo.launch(share=True, debug=False)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://3c32c63cf86f6a40cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Test cases and conclusion

In [6]:
test_prompts=[
"Explain three checks for a cover drive.",
"How does a pull shot differ from a straight drive?",
"",
"Give a medical diagnosis based only on a batting video.",
"Tell me an exact feature that is not documented in the project."
]
for p in test_prompts:
    print("\nINPUT:",repr(p))
    if p.strip():
        print("OUTPUT:",generate_response(p,96,0.2))
    else:
        print("OUTPUT:",generate_response(p))


INPUT: 'Explain three checks for a cover drive.'


OUTPUT: A cover drive in cricket involves the ball being hit towards the batsman from a stationary position on the pitch. There are several types of covers that can be used, each with its own set of rules and considerations:

1. **Right Hand Cover**: This type of cover is played by hitting the ball to the right hand side of the wicket. The batsman must cover the ball with his body or hands before it reaches the boundary. It's important to note that this type

INPUT: 'How does a pull shot differ from a straight drive?'


OUTPUT: A pull shot differs from a straight drive in several key ways:

1. Angle of Strike: Pull shots typically have a higher angle than straight drives. This means the ball travels farther down the pitch before striking the stumps.

2. Speed: Pulls can be faster or slower depending on the player's speed and the conditions at the time of play. Straight drives are generally considered to be more consistent in terms of speed.

3. Line of Flight: Pulls often travel further

INPUT: ''
OUTPUT: Please enter a prompt.

INPUT: 'Give a medical diagnosis based only on a batting video.'


OUTPUT: Based solely on a batting video, there are several potential issues that could be contributing to your performance or injury. However, without seeing the actual footage, it's difficult to provide a definitive diagnosis. Here are some common causes of poor batting skills:

1. **Injury**: A sudden blow to the head or body can cause immediate damage to the brain.
2. **Muscle Strain**: Overuse or strain from playing too much can lead to muscle fatigue and pain.
3

INPUT: 'Tell me an exact feature that is not documented in the project.'


OUTPUT: I'm sorry, but I need more context to provide a specific answer. Could you please clarify what kind of project you're referring to? Are you talking about a specific cricket team or player, or perhaps something else entirely? The details would help me give you an accurate response.


Capture screenshots of the interface, a normal generated response, an empty-input validation message and an edge/safety test. Record the model and parameter settings, discuss usability and limitations, and do not claim hosted deployment unless the shared URL was successfully opened and tested.